In [ ]:
# ============================================================
# STEP 1: Setup Kaggle API
# ============================================================
import os

# Upload your kaggle.json from your system (download it from Kaggle account settings)
from google.colab import files
files.upload()   # <- Upload kaggle.json here

# Move kaggle.json to the correct location
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# ============================================================
# STEP 2: Download Fruits360 dataset
# ============================================================
# Dataset link: https://www.kaggle.com/datasets/moltean/fruits
!kaggle datasets download -d moltean/fruits -p /content --unzip

# Check extracted folders
!ls /content/fruits-360-original-size

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/moltean/fruits
License(s): CC-BY-SA-4.0
100% 4.30G/4.31G [01:11<00:00, 30.8MB/s]
100% 4.31G/4.31G [01:11<00:00, 64.4MB/s]
ls: cannot access '/content/fruits-360-original-size': No such file or directory


In [ ]:
# Model Genrator Code.
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# -------------------
# 1. Transformations
# -------------------
train_tfms = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])  # ImageNet stats
])

val_tfms = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# -------------------
# 2. Dataset Paths
# -------------------
train_dir = "/kaggle/input/fruits/fruits-360_original-size/fruits-360-original-size/Training"
val_dir   = "/kaggle/input/fruits/fruits-360_original-size/fruits-360-original-size/Validation"
test_dir  = "/kaggle/input/fruits/fruits-360_original-size/fruits-360-original-size/Test"

train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
val_ds   = datasets.ImageFolder(val_dir, transform=val_tfms)
test_ds  = datasets.ImageFolder(test_dir, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)

# -------------------
# 3. Model (EfficientNetB0)
# -------------------
model = models.mobilenet_v2(pretrained=True, width_mult=0.35)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(train_ds.classes))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# -------------------
# 4. Loss & Optimizer
# -------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# -------------------
# 5. Training Loop
# -------------------
def train_model(model, criterion, optimizer, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

        train_acc = 100. * correct / total

        # Validation
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, preds = outputs.max(1)
                correct += preds.eq(labels).sum().item()
                total += labels.size(0)

        val_acc = 100. * correct / total

        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}% "
              f"Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.2f}%")

    return model

model = train_model(model, criterion, optimizer, train_loader, val_loader, epochs=10)

# -------------------
# 6. Save Model
# -------------------
torch.save(model.state_dict(), "efficientnet_fruits360.pth")

# -------------------
# 7. Final Test Evaluation
# -------------------
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100. * correct / total:.2f}%")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/fruits/fruits-360_original-size/fruits-360-original-size/Training'

# MobileNetV2

In [ ]:
# -------------------
# MobileNetV2 for Fruits360
# -------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# -------------------
# 1. Transformations
# -------------------
train_tfms = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])  # ImageNet stats
])

val_tfms = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# -------------------
# 2. Dataset Paths
# -------------------
train_dir = "/content/fruits-360_original-size/fruits-360-original-size/Training"
val_dir   = "/content/fruits-360_original-size/fruits-360-original-size/Validation"
test_dir  = "/content/fruits-360_original-size/fruits-360-original-size/Test"

train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
val_ds   = datasets.ImageFolder(val_dir, transform=val_tfms)
test_ds  = datasets.ImageFolder(test_dir, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)

# -------------------
# 3. Model (MobileNetV2)
# -------------------
model = models.mobilenet_v2(weights=None, width_mult=0.35)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(train_ds.classes))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# -------------------
# 4. Loss & Optimizer
# -------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# -------------------
# 5. Training Loop
# -------------------
def train_model(model, criterion, optimizer, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

        train_acc = 100. * correct / total

        # Validation
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, preds = outputs.max(1)
                correct += preds.eq(labels).sum().item()
                total += labels.size(0)

        val_acc = 100. * correct / total

        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}% "
              f"Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.2f}%")

    return model

model = train_model(model, criterion, optimizer, train_loader, val_loader, epochs=10)

# -------------------
# 6. Save Model
# -------------------
torch.save(model.state_dict(), "mobilenetv2_fruits360.pth")

# -------------------
# 7. Final Test Evaluation
# -------------------
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100. * correct / total:.2f}%")

Epoch [1/10] Train Loss: 3.7844, Train Acc: 10.49% Val Loss: 2.6424, Val Acc: 29.38%
Epoch [2/10] Train Loss: 1.9603, Train Acc: 45.17% Val Loss: 1.1385, Val Acc: 69.08%
Epoch [3/10] Train Loss: 0.9771, Train Acc: 72.02% Val Loss: 0.5442, Val Acc: 85.49%
Epoch [4/10] Train Loss: 0.5351, Train Acc: 85.32% Val Loss: 0.2618, Val Acc: 93.93%
Epoch [5/10] Train Loss: 0.3115, Train Acc: 91.83% Val Loss: 0.1321, Val Acc: 97.48%
Epoch [6/10] Train Loss: 0.2040, Train Acc: 94.76% Val Loss: 0.0837, Val Acc: 98.44%
Epoch [7/10] Train Loss: 0.1277, Train Acc: 96.91% Val Loss: 0.0402, Val Acc: 99.44%
Epoch [8/10] Train Loss: 0.0931, Train Acc: 97.85% Val Loss: 0.0249, Val Acc: 99.73%
Epoch [9/10] Train Loss: 0.0639, Train Acc: 98.66% Val Loss: 0.0141, Val Acc: 99.86%
Epoch [10/10] Train Loss: 0.0517, Train Acc: 98.82% Val Loss: 0.0176, Val Acc: 99.68%
Test Accuracy: 99.77%


In [ ]:
# ====================================
# Cell 0 — Load the trained model from .pth
# ====================================
import torch
import torch.nn as nn
from torchvision import models

# Set the device to load the model on
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the trained weights first to inspect them
model_path = "/content/mobilenetv2_fruits360.pth"
state_dict = torch.load(model_path, map_location=device)

# Dynamically determine the number of classes from the loaded state dictionary
# The number of classes is the first dimension of the final classifier's weight tensor
# For MobileNetV2, the classifier is a Sequential with Linear layer at index 1
num_classes = state_dict['classifier.1.weight'].shape[0]
print(f"✅ Found {num_classes} classes in the loaded model weights.")

# Re-create the model architecture with the correct number of classes and width_mult
model = models.mobilenet_v2(weights=None, width_mult=0.35) # Use weights=None to load custom weights
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, num_classes)
model.to(device)

# Load the weights into the correctly-sized model
model.load_state_dict(state_dict)

print("✅ PyTorch model loaded successfully with correct classifier size.")

✅ Found 104 classes in the loaded model weights.
✅ PyTorch model loaded successfully with correct classifier size.


In [ ]:
!pip install onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 113.0 MB/s eta 0:00:00


In [ ]:
# Step 1: Export PyTorch → ONNX
# ====================================
# Cell 1 — Export loaded PyTorch model to ONNX
# ====================================
import torch

# Ensure the model is in evaluation mode for proper export
model.eval()

# Create a dummy input tensor with the correct shape (1, 3, 100, 100)
# The first dimension (1) is the batch size.
dummy_input = torch.randn(1, 3, 100, 100).to(device)

# Export the model to ONNX
# Using opset_version=13 ensures broad compatibility.
torch.onnx.export(model,
                  dummy_input,
                  "mobilenetv2_fruits360.onnx",
                  input_names=["input_1"],
                  output_names=["output_0"],
                  dynamic_axes={"input_1": {0: "batch_size"},
                                "output_0": {0: "batch_size"}},
                  opset_version=13)

print("✅ ONNX model saved successfully!")

/tmp/ipython-input-4011427852.py:16: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(model,


✅ ONNX model saved successfully!


In [ ]:
# Check the ONNX model input/outputs
import onnx

model = onnx.load("/content/mobilenetv2_fruits360.onnx")

print("Inputs:")
for inp in model.graph.input:
    print(f"  {inp.name} -> {[d.dim_value for d in inp.type.tensor_type.shape.dim]}")

print("\nOutputs:")
for out in model.graph.output:
    print(f"  {out.name} -> {[d.dim_value for d in out.type.tensor_type.shape.dim]}")

Inputs:
  input_1 -> [0, 3, 100, 100]

Outputs:
  output_0 -> [0, 104]


In [ ]:
!pip install onnxruntime

import onnx

# Load ONNX model
onnx_model = onnx.load("/content/mobilenetv2_fruits360.onnx")

# Print input and output info
print("== Model Inputs ==")
for i in onnx_model.graph.input:
    print(f"Name: {i.name}, Shape: {[d.dim_value for d in i.type.tensor_type.shape.dim]}, Type: {i.type.tensor_type.elem_type}")

print("\n== Model Outputs ==")
for o in onnx_model.graph.output:
    print(f"Name: {o.name}, Shape: {[d.dim_value for d in o.type.tensor_type.shape.dim]}, Type: {o.type.tensor_type.elem_type}")

print("\n== Opset Version ==")
print(onnx_model.opset_import[0].version)

# List first 20 ops to see if we need TensorFlow Addons
print("\n== First 20 nodes ==")
for node in onnx_model.graph.node[:20]:
    print(node.op_type)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.5 MB/s eta 0:00:00
== Model Inputs ==
Name: input_1, Shape: [0, 3, 100, 100], Type: 1

== Model Outputs ==
Name: output_0, Shape: [0, 104], Type: 1

== Opset Version ==
13

== First 20 nodes ==
Conv
Constant
Constant
Clip
Conv
Constant
Constant
Clip
Conv
Conv
Constant
Constant
Clip
Conv
Constant
Constant
Clip
Conv
Conv
Constant


In [ ]:
# → ONNX → TensorFlow → TFLite (float32/float16)
# Install all dependencies properly
!pip install onnx2tf onnx-graphsurgeon ai-edge-litert sng4onnx

# Imports
import onnx
import tensorflow as tf
from onnx2tf import convert

# Path to your ONNX model
onnx_model_path = "/content/mobilenetv2_fruits360.onnx"
output_dir = "/content/tf_model"

# Convert ONNX → TensorFlow (SavedModel + TFLite)
convert(
    input_onnx_file_path=onnx_model_path,
    output_folder_path=output_dir,
    copy_onnx_input_output_names_to_tflite=True,
    non_verbose=True
)

print("✅ Conversion completed. Model saved to:", output_dir)

# (Optional) Explicitly export TFLite file
converter = tf.lite.TFLiteConverter.from_saved_model(output_dir)
tflite_model = converter.convert()
with open("/content/mobilenetv2_fruits360.tflite", "wb") as f:
    f.write(tflite_model)

print("✅ TFLite model exported:", "/content/mobilenetv2_fruits360.tflite")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.3/468.3 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 MB 8.1 MB/s eta 0:00:00
✅ Conversion completed. Model saved to: /content/tf_model
✅ TFLite model exported: /content/mobilenetv2_fruits360.tflite


In [ ]:
import os
print("Float 16 TFLite model size (MB):", os.path.getsize("/content/tf_model/mobilenetv2_fruits360_float16.tflite")/1024/1024)
print("Float 32 TFLite model size (MB):", os.path.getsize("/content/tf_model/mobilenetv2_fruits360_float32.tflite")/1024/1024)

Float 16 TFLite model size (MB): 1.0323104858398438
Float 32 TFLite model size (MB): 2.0169677734375


In [ ]:
# → Create Proper Calibration Dataset
# ============================================================
# STEP 1: Extract Training folder only
# ============================================================
import shutil

train_dir = "/content/fruits-360_original-size/fruits-360-original-size/Training"
sample_dir = "/content/calibration_samples"

os.makedirs(sample_dir, exist_ok=True)

# ============================================================
# STEP 2: Sample ~200 images across classes for calibration
# ============================================================
import random
import glob

all_classes = os.listdir(train_dir)
print("Total classes:", len(all_classes))

# Pick ~10 random images per class (to total ~200, adjust if needed)
images_per_class = max(1, 200 // len(all_classes))

for cls in all_classes:
    cls_path = os.path.join(train_dir, cls)
    cls_images = glob.glob(os.path.join(cls_path, "*.jpg"))
    sampled = random.sample(cls_images, min(images_per_class, len(cls_images)))

    # Make class folder inside calibration samples
    out_cls_path = os.path.join(sample_dir, cls)
    os.makedirs(out_cls_path, exist_ok=True)

    # Copy sampled images
    for img in sampled:
        shutil.copy(img, out_cls_path)

print("✅ Calibration dataset created at:", sample_dir)

# Count images
total = sum(len(files) for _, _, files in os.walk(sample_dir))
print("Total calibration images:", total)

Total classes: 104
✅ Calibration dataset created at: /content/calibration_samples
Total calibration images: 104


In [ ]:
# → INT8 Quantization (using calibration_samples)
import tensorflow as tf
import numpy as np
from PIL import Image
import os

# Path to your already converted Float32 model
saved_model_dir = "/content/tf_model"   # ✅ fix here
tflite_int8_path = "/content/mobilenetv2_fruits360_int-8.tflite"

# ============================================================
# Step 1: Define representative dataset generator
# ============================================================
def representative_dataset_gen():
    calib_images = []
    for root, _, files in os.walk("/content/calibration_samples"):
        for file in files:
            if file.lower().endswith(".jpg"):
                calib_images.append(os.path.join(root, file))

    for img_path in calib_images:
        img = Image.open(img_path).resize((100, 100))  # adjust to your model input
        img = np.array(img).astype(np.float32) / 255.0
        img = np.expand_dims(img, axis=0)  # add batch dimension
        yield [img]

# ============================================================
# Step 2: Convert to INT8 TFLite model
# ============================================================
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen

# Force full integer quantization (weights + activations)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_int8_model = converter.convert()

# Save INT8 model
with open(tflite_int8_path, "wb") as f:
    f.write(tflite_int8_model)

print("✅ INT8 Quantized model saved at:", tflite_int8_path)

✅ INT8 Quantized model saved at: /content/mobilenetv2_fruits360_int-8.tflite


In [ ]:
# → Sanity Check (FP32 vs INT8 Inference)
import tensorflow as tf
import numpy as np
from PIL import Image
import os

# Paths to models
fp32_path = "/content/tf_model/mobilenetv2_fruits360_float32.tflite"
int8_path = "/content/mobilenetv2_fruits360_int-8.tflite"

# Load both interpreters
interp_fp32 = tf.lite.Interpreter(model_path=fp32_path)
interp_int8 = tf.lite.Interpreter(model_path=int8_path)
interp_fp32.allocate_tensors()
interp_int8.allocate_tensors()

# Input details
in_details_fp32 = interp_fp32.get_input_details()
in_details_int8 = interp_int8.get_input_details()
out_details_fp32 = interp_fp32.get_output_details()
out_details_int8 = interp_int8.get_output_details()

# Helper: preprocess
def preprocess(img_path):
    img = Image.open(img_path).resize((100, 100))
    arr = np.array(img).astype(np.float32) / 255.0
    return np.expand_dims(arr, axis=0)

# Pick ~10 random images from calibration set
import random
calib_images = []
for root, _, files in os.walk("/content/calibration_samples"):
    for file in files:
        if file.endswith(".jpg"):
            calib_images.append(os.path.join(root, file))
sample_imgs = random.sample(calib_images, 10)

print("🔍 Sanity check on", len(sample_imgs), "images...\n")

for img_path in sample_imgs:
    x = preprocess(img_path)

    # FP32 inference
    interp_fp32.set_tensor(in_details_fp32[0]['index'], x.astype(np.float32))
    interp_fp32.invoke()
    out_fp32 = interp_fp32.get_tensor(out_details_fp32[0]['index'])[0]

    # INT8 inference
    # Scale input to int8 if required
    scale, zero_point = in_details_int8[0]['quantization']
    x_int8 = x / scale + zero_point
    x_int8 = np.clip(x_int8, -128, 127).astype(np.int8)
    interp_int8.set_tensor(in_details_int8[0]['index'], x_int8)
    interp_int8.invoke()
    out_int8 = interp_int8.get_tensor(out_details_int8[0]['index'])[0]

    # Compare top-3 predictions
    top3_fp32 = np.argsort(out_fp32)[-3:][::-1]
    top3_int8 = np.argsort(out_int8)[-3:][::-1]

    print(f"🖼️ {os.path.basename(img_path)}")
    print(" FP32 top3:", top3_fp32, "scores:", out_fp32[top3_fp32])
    print(" INT8 top3:", top3_int8, "scores:", out_int8[top3_int8])
    print(" Match?   ", top3_fp32[0] == top3_int8[0])
    print("-"*50)

🔍 Sanity check on 10 images...

🖼️ r1_266.jpg
 FP32 top3: [29 32 34] scores: [6.8022294 5.380168  4.9311867]
 INT8 top3: [29 34 32] scores: [99 85 82]
 Match?    True
--------------------------------------------------
🖼️ r2_124.jpg
 FP32 top3: [29 34 35] scores: [6.390599  5.3624954 5.154653 ]
 INT8 top3: [29 34 35] scores: [96 89 80]
 Match?    True
--------------------------------------------------
🖼️ r0_158.jpg
 FP32 top3: [29 35 32] scores: [4.573078  4.4041524 3.4968984]
 INT8 top3: [29 35 32] scores: [79 72 67]
 Match?    True
--------------------------------------------------
🖼️ r2_314.jpg
 FP32 top3: [32 29 35] scores: [4.645565  4.5669775 3.6294763]
 INT8 top3: [32 29 35] scores: [74 74 64]
 Match?    True
--------------------------------------------------
🖼️ r1_98.jpg
 FP32 top3: [29 32 35] scores: [8.295941  5.305763  5.1060705]
 INT8 top3: [29 35 32] scores: [111  83  83]
 Match?    True
--------------------------------------------------
🖼️ r0_298.jpg
 FP32 top3: [32 34 29]

In [ ]:
# → Compare Model Sizes
import os

# Paths to your models
fp32_path = "/content/tf_model/mobilenetv2_fruits360_float32.tflite"
fp16_path = "/content/tf_model/mobilenetv2_fruits360_float16.tflite"
int8_path = "/content/mobilenetv2_fruits360_int-8.tflite"

def file_size(path):
    size_mb = os.path.getsize(path) / (1024 * 1024)
    return round(size_mb, 3)

print("📦 Model Sizes:")
print(f"Float32 model: {file_size(fp32_path)} MB")
print(f"Float16 model: {file_size(fp16_path)} MB")
print(f"INT8 model:    {file_size(int8_path)} MB")

📦 Model Sizes:
Float32 model: 2.017 MB
Float16 model: 1.032 MB
INT8 model:    0.712 MB


In [ ]:
# → Convert INT8 TFLite → C array (ESP32 deployment)
# ============================================================
# Convert TFLite model into a C array for ESP32
# ============================================================
import pathlib

tflite_model_path = "/content/mobilenetv2_fruits360_int-8.tflite"
tflite_model_c_path = "/content/mobilenetv2_fruits360_int8.cpp"

with open(tflite_model_path, "rb") as f:
    tflite_data = f.read()

hex_array = ", ".join("0x{:02x}".format(x) for x in tflite_data)

c_file_content = f"""
#include "tensorflow/lite/micro/micro_mutable_op_resolver.h"
#include "tensorflow/lite/micro/micro_interpreter.h"
#include "tensorflow/lite/schema/schema_generated.h"
#include "tensorflow/lite/version.h"

const unsigned char mobilenetv2_fruits360_int8[] = {{
{hex_array}
}};

const int mobilenetv2_fruits360_int8_len = {len(tflite_data)};
"""

with open(tflite_model_c_path, "w") as f:
    f.write(c_file_content)

print("✅ C array file saved:", tflite_model_c_path)

✅ C array file saved: /content/mobilenetv2_fruits360_int8.cpp


In [ ]:
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
import os

# Input / output paths
model_fp32 = "/content/efficientnet_fruits360.onnx"   # original ONNX
model_int8 = "model-requant.onnx"

# Apply dynamic quantization (weights → INT8)
quantize_dynamic(
    model_input=model_fp32,
    model_output=model_int8,
    weight_type=QuantType.QInt8  # quantize weights to INT8
)

# Print size comparison
orig_size = os.path.getsize(model_fp32) / 1024 / 1024
quant_size = os.path.getsize(model_int8) / 1024 / 1024
print(f"📦 Original model size: {orig_size:.2f} MB")
print(f"✅ Re-Quantized model size: {quant_size:.2f} MB")


ValidationError: Unable to open proto file: /content/efficientnet_fruits360.onnx. Please check if it is a valid proto. 